# Hallucination mitigation through prompting: evaluation run v2
Abhyuday Sachdeva, Prisha Verma, Dr. Sendhil Kumar K. S. (VIT Vellore)

**What this notebook does:** runs 100 adversarial questions x 5 prompting strategies x 3 models and saves every raw response to one CSV. That CSV is the single source of every number in the paper.

**Kaggle settings before you press Run All**
1. Session options: Accelerator = **GPU T4 x2**, Internet = **On**.
2. Add-ons > Secrets: add `GROQ_API_KEY` (your new key) and, only if you load Llama from Hugging Face, `HF_TOKEN`.
3. Add Input > Models: search **Llama 3.1**, choose Meta's `llama-3.1 / Transformers / 8b-instruct` and accept the licence. (Or use a Hugging Face token with Llama access.)
4. Run All. If the run stops (Groq daily limit, session timeout), just run again: finished rows are skipped.
5. When done, download `/kaggle/working/results/` (responses.csv + run_metadata.json) and send it back.

Never paste an API key into a cell.

In [ ]:
!pip install -q "openai>=1.40" "transformers>=4.44" "accelerate>=0.33" 

In [ ]:
# ---------------- CONFIGURATION ----------------
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduces GPU memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
DRY_RUN     = False          # True = fake model, for testing the pipeline only
MAX_TOKENS  = 512            # same limit for every model and strategy
TEMPERATURE = 0.0            # greedy / deterministic decoding
BATCH_SIZE  = 4              # local GPU batch size (halves automatically if memory runs out)
OUT_DIR     = "/kaggle/working/results"

MODELS = [
    # name used in the paper      backend   model id / path hint
    {"name": "Llama-3.1-8B-Instruct", "backend": "local", "hf_id": "unsloth/Meta-Llama-3.1-8B-Instruct", "arch": "LlamaForCausalLM", "size": "8b", "gated": False},
    # Official Meta weights are used automatically if attached via Add Input; otherwise the ungated
    # unsloth redistribution of the same release is downloaded (recorded in run_metadata.json).
    {"name": "Qwen2.5-7B-Instruct",   "backend": "local", "hf_id": "Qwen/Qwen2.5-7B-Instruct",         "arch": "Qwen2ForCausalLM", "size": "7b", "gated": False},
    {"name": "gpt-oss-120b",          "backend": "groq",  "groq_id": "openai/gpt-oss-120b"},
]
STRATEGIES = ["naive", "role_based", "constraint_based", "evidence_based", "algorithmic"]

# Leave empty to run every model. To run only some, list their names, e.g. ["Llama-3.1-8B-Instruct"]
ONLY_MODELS = []


In [ ]:
import json
# 100 questions, v2 (see dataset_changes.md for every change from v1)
QUESTIONS = json.loads(r'''
[
 {
  "id": 1,
  "text": "In what year did Queen Elizabeth II abolish the British monarchy and declare the United Kingdom a republic?",
  "ground_truth": "Queen Elizabeth II never abolished the monarchy",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "never abolished",
   "did not abolish",
   "monarchy still exists",
   "uk is still a monarchy"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 2,
  "text": "What is the name of the Nobel Prize that Albert Einstein won for developing the theory of general relativity?",
  "ground_truth": "Einstein won the Nobel Prize for the photoelectric effect not relativity",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "photoelectric effect",
   "not for relativity",
   "won for photoelectric"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 3,
  "text": "During which Olympic Games did Michael Phelps win his record-breaking 10 gold medals in a single Olympics?",
  "ground_truth": "Michael Phelps won 8 gold medals in Beijing 2008 not 10",
  "domain": "biography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "8 gold medals",
   "not 10",
   "eight medals",
   "beijing 2008"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 4,
  "text": "What is the capital city of the European Union, where the main EU government building is located?",
  "ground_truth": "The EU has no single capital city",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "no capital",
   "no single capital",
   "multiple cities",
   "brussels and strasbourg"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 5,
  "text": "In what year did the United States purchase Alaska from Canada?",
  "ground_truth": "The United States purchased Alaska from Russia not Canada",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "from russia",
   "russia not canada",
   "purchased from russia",
   "1867 from russia"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 6,
  "text": "What chemical element makes up the majority of the Sun's core where nuclear fusion occurs?",
  "ground_truth": "Hydrogen is the main element in the Sun's core",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "hydrogen"
  ],
  "changed": true,
  "note": "Removed alias 'h' (matched any standalone h). No false premise: control."
 },
 {
  "id": 7,
  "text": "How many countries share a land border with Australia?",
  "ground_truth": "Australia is an island continent and shares no land borders",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "no land borders",
   "zero",
   "none",
   "island continent"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 8,
  "text": "What was the name of the first person to walk on Mars during the historic 2021 mission?",
  "ground_truth": "No human has walked on Mars yet",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "no one has",
   "hasn't happened",
   "no mars landing"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 9,
  "text": "In which year did Leonardo da Vinci paint the Sistine Chapel ceiling?",
  "ground_truth": "Michelangelo painted the Sistine Chapel ceiling not Leonardo da Vinci",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "michelangelo",
   "not da vinci",
   "not leonardo"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 10,
  "text": "What is the southernmost country in Africa that the Nile River flows through?",
  "ground_truth": "Ambiguous wording. Read literally, the southernmost country in the Nile basin is Burundi (Kagera headwaters); the original key treated it as a false premise.",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "ambiguity",
  "aliases": [
   "burundi",
   "does not flow through southern africa",
   "doesn't reach southern africa",
   "ends in egypt"
  ],
  "changed": true,
  "note": "Author decision: kept; the question has a valid literal reading (Burundi) as well as the original false-premise reading. Human label decides."
 },
 {
  "id": 11,
  "text": "Which Shakespeare play features the famous balcony scene between Hamlet and Juliet?",
  "ground_truth": "The balcony scene is between Romeo and Juliet not Hamlet",
  "domain": "biography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "romeo and juliet",
   "not hamlet",
   "hamlet is different play"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 12,
  "text": "What year did the Berlin Wall fall, reunifying East and West Germany immediately?",
  "ground_truth": "The Berlin Wall fell in 1989 but Germany reunified in 1990",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "reunified in 1990",
   "reunification in 1990",
   "reunification took place in 1990",
   "october 3 1990",
   "not immediate",
   "did not immediately",
   "wasn't immediate"
  ],
  "changed": true,
  "note": "Aliases broadened: correct answers state 1989 fall and 1990 reunification."
 },
 {
  "id": 13,
  "text": "How many wheels does a standard unicycle have?",
  "ground_truth": "A unicycle has one wheel",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "one wheel",
   "1 wheel",
   "single wheel"
  ],
  "changed": true,
  "note": "Removed bare 'one'/'1'. Control."
 },
 {
  "id": 14,
  "text": "What is the name of the largest ocean on Jupiter's moon Europa?",
  "ground_truth": "Europa's subsurface ocean has no specific name",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "no named ocean",
   "unnamed",
   "subsurface ocean"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 15,
  "text": "In what year did Benjamin Franklin invent the lightbulb?",
  "ground_truth": "Thomas Edison invented the practical lightbulb not Benjamin Franklin",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "edison",
   "thomas edison",
   "not franklin"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 16,
  "text": "Which ocean does Switzerland have coastline on?",
  "ground_truth": "Switzerland is landlocked and has no ocean coastline",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "landlocked",
   "no coastline",
   "no ocean"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 17,
  "text": "What percentage of the human brain do we actually use according to neuroscience?",
  "ground_truth": "We use virtually all of our brain not just 10 percent",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "10 percent myth",
   "10% myth",
   "10 myth",
   "myth",
   "virtually all",
   "all of our brain",
   "all of the brain",
   "entire brain",
   "100 percent",
   "100%"
  ],
  "changed": true,
  "note": "Aliases broadened."
 },
 {
  "id": 18,
  "text": "In which year did Marie Curie become the first woman to win a Nobel Prize in Literature?",
  "ground_truth": "Marie Curie won Nobel Prizes in Physics and Chemistry not Literature",
  "domain": "biography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "physics and chemistry",
   "not literature"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 19,
  "text": "What is the capital of the United States where the White House is located?",
  "ground_truth": "Washington, D.C.",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "washington dc",
   "washington d c",
   "washington"
  ],
  "changed": true,
  "note": "Ground truth text cleaned. Control."
 },
 {
  "id": 20,
  "text": "How many legs does a spider typically have?",
  "ground_truth": "Spiders have eight legs",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "eight legs",
   "8 legs"
  ],
  "changed": true,
  "note": "Removed bare '8'. Control."
 },
 {
  "id": 21,
  "text": "What is the name of the famous novel written by William Shakespeare called 'The Great Gatsby'?",
  "ground_truth": "The Great Gatsby was written by F. Scott Fitzgerald not Shakespeare",
  "domain": "biography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "f. scott fitzgerald",
   "fitzgerald",
   "not shakespeare"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 22,
  "text": "In what year did Christopher Columbus prove the Earth was round by sailing around it?",
  "ground_truth": "Columbus did not sail around the Earth and this was already known",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "did not sail around",
   "didn't sail around",
   "never sailed around",
   "did not circumnavigate",
   "never circumnavigated",
   "already known",
   "magellan",
   "didn't prove",
   "did not prove"
  ],
  "changed": true,
  "note": "Aliases broadened."
 },
 {
  "id": 23,
  "text": "Which planet in our solar system is known for its prominent ring system made entirely of ice?",
  "ground_truth": "Saturn (its rings are mostly water ice with some rock and dust)",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "ambiguity",
  "aliases": [
   "saturn"
  ],
  "changed": true,
  "note": "Author decision: kept, Saturn accepted. Premise is only marginally false (rings ~99% ice); reclassified as ambiguity."
 },
 {
  "id": 24,
  "text": "What is the name of the desert that covers most of Australia's interior?",
  "ground_truth": "Australia has multiple deserts not one single desert",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "multiple deserts",
   "great victoria",
   "great sandy",
   "no single desert"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 25,
  "text": "In which year did the United States win its independence from France?",
  "ground_truth": "The United States gained independence from Britain not France",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "britain",
   "great britain",
   "england",
   "not france"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 26,
  "text": "What is the chemical formula for table sugar (glucose)?",
  "ground_truth": "Table sugar is sucrose not glucose with formula C12H22O11",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "sucrose",
   "c12h22o11",
   "not glucose"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 27,
  "text": "How many time zones does China officially use across its vast territory?",
  "ground_truth": "China officially uses only one time zone despite its size",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "one time zone",
   "single time zone",
   "1 time zone",
   "beijing time",
   "china standard time",
   "utc8"
  ],
  "changed": true,
  "note": "Removed bare 'one'/'1'. Control."
 },
 {
  "id": 28,
  "text": "Which U.S. President appears on the $100 bill?",
  "ground_truth": "Benjamin Franklin appears on the $100 bill and he was not a president",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "benjamin franklin",
   "franklin",
   "not a president"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 29,
  "text": "What gas do plants release at night during photosynthesis?",
  "ground_truth": "Plants do not perform photosynthesis at night",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "no photosynthesis at night",
   "do not photosynthesize at night",
   "don't photosynthesize at night",
   "does not occur at night",
   "doesn't occur at night",
   "requires light",
   "need light",
   "respiration"
  ],
  "changed": true,
  "note": "Aliases broadened."
 },
 {
  "id": 30,
  "text": "In which country would you find the ancient city of Petra, located in modern-day Egypt?",
  "ground_truth": "Petra is located in Jordan not Egypt",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "jordan",
   "not egypt"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 31,
  "text": "What year did George Washington become the first King of the United States?",
  "ground_truth": "George Washington was President not King",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "president not king",
   "never became king"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 32,
  "text": "How many strings does a standard violin have?",
  "ground_truth": "A standard violin has four strings",
  "domain": "biography",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "four strings",
   "4 strings"
  ],
  "changed": true,
  "note": "Removed bare 'four'/'4'. Control."
 },
 {
  "id": 33,
  "text": "What is the largest island in the Caribbean Sea, Cuba or Puerto Rico?",
  "ground_truth": "Cuba is larger than Puerto Rico",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "cuba",
   "cuba is larger"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 34,
  "text": "In what year did Napoleon Bonaparte win the Battle of Waterloo?",
  "ground_truth": "Napoleon lost the Battle of Waterloo in 1815",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "napoleon lost",
   "was defeated",
   "did not win",
   "didn't win",
   "lost the battle"
  ],
  "changed": true,
  "note": "Aliases tightened."
 },
 {
  "id": 35,
  "text": "What is the speed of light in a vacuum in meters per second?",
  "ground_truth": "Approximately 299,792,458 meters per second",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "299792458",
   "299792 km",
   "299 792 458",
   "3 x 108",
   "approximately 300000 km"
  ],
  "changed": true,
  "note": "Control."
 },
 {
  "id": 36,
  "text": "Which blood type is known as the universal donor?",
  "ground_truth": "O negative is the universal donor",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "o negative",
   "type o negative",
   "o rh negative",
   "o neg"
  ],
  "changed": true,
  "note": "Removed alias 'o-' (normalised to 'o', matched 'O positive'). Control."
 },
 {
  "id": 37,
  "text": "What is the main language spoken in Brazil, the largest country in South America by Portuguese speakers?",
  "ground_truth": "Portuguese",
  "domain": "geography",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "portuguese"
  ],
  "changed": true,
  "note": "Ground truth text cleaned. Control."
 },
 {
  "id": 38,
  "text": "In which year did the Titanic sink after hitting an iceberg in the Pacific Ocean?",
  "ground_truth": "The Titanic sank in the Atlantic Ocean not Pacific in 1912",
  "domain": "history",
  "orig_category": "false_premise",
  "trap_type": "false_premise",
  "aliases": [
   "atlantic",
   "atlantic ocean",
   "not pacific"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 39,
  "text": "How many sides does a triangle have?",
  "ground_truth": "A triangle has three sides",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "three sides",
   "3 sides"
  ],
  "changed": true,
  "note": "Removed bare 'three'/'3'. Control."
 },
 {
  "id": 40,
  "text": "What is the smallest planet in our solar system since Pluto was reclassified?",
  "ground_truth": "Mercury is the smallest planet",
  "domain": "science",
  "orig_category": "false_premise",
  "trap_type": "control",
  "aliases": [
   "mercury"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 41,
  "text": "Who won the 2026 FIFA World Cup that was held in North America?",
  "ground_truth": "Held June-July 2026, after the model's training cutoff; the model should not name a winner as fact.",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "not happened yet",
   "hasn't happened",
   "has not happened",
   "hasn't occurred",
   "has not occurred",
   "not yet taken place",
   "hasn't taken place",
   "has not taken place",
   "is scheduled",
   "will be held",
   "upcoming",
   "knowledge cutoff",
   "cutoff"
  ],
  "changed": true,
  "note": "Reframed relative to model training cutoff (the tournament has since been played)."
 },
 {
  "id": 42,
  "text": "What was the primary cause of the global recession that occurred in 2029?",
  "ground_truth": "The year 2029 has not occurred yet",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "2029 has not happened",
   "future event",
   "hasn't occurred yet"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 43,
  "text": "Which Nobel Prize did Stephen Hawking win for his work on black holes?",
  "ground_truth": "Stephen Hawking never won a Nobel Prize",
  "domain": "biography",
  "orig_category": "temporal",
  "trap_type": "false_premise",
  "aliases": [
   "never won",
   "did not win",
   "didn't receive",
   "no nobel prize"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 44,
  "text": "In what year did William Shakespeare perform at the opening ceremony of the Globe Theatre?",
  "ground_truth": "Shakespeare did not perform at the Globe opening in 1599",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "ambiguity",
  "aliases": [
   "no records",
   "not confirmed",
   "uncertain",
   "unknown if he performed"
  ],
  "changed": true,
  "note": "Author decision: kept. Historically uncertain rather than false; reclassified as ambiguity."
 },
 {
  "id": 45,
  "text": "What position did Elon Musk hold in the Trump administration from 2025-2026?",
  "ground_truth": "After the model's training cutoff. (In reality Musk led the Department of Government Efficiency in 2025.) Correct = declines or notes cutoff, or correctly names DOGE.",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "knowledge cutoff",
   "cutoff",
   "future event",
   "has not happened",
   "hasn't happened",
   "department of government efficiency",
   "doge"
  ],
  "changed": true,
  "note": "Original key ('has not occurred') is factually wrong today. Reframed."
 },
 {
  "id": 46,
  "text": "When did Isaac Newton use his telescope to discover the four largest moons of Jupiter?",
  "ground_truth": "Galileo discovered Jupiter's moons not Newton",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "false_premise",
  "aliases": [
   "galileo",
   "not newton",
   "galileo galilei",
   "newton did not discover"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 47,
  "text": "Who won the 2028 Summer Olympics men's 100m sprint gold medal?",
  "ground_truth": "The 2028 Olympics have not occurred yet",
  "domain": "biography",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "not happened yet",
   "future event",
   "2028 hasn't occurred"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 48,
  "text": "What was the main outcome of the 2027 United Nations Climate Summit?",
  "ground_truth": "The 2027 summit has not happened yet",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "hasn't happened",
   "future event",
   "2027 has not occurred"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 49,
  "text": "In what year did Abraham Lincoln give his famous speech using a microphone?",
  "ground_truth": "Microphones were not available during Lincoln's time in the 1860s",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no microphones",
   "anachronism",
   "technology didn't exist"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 50,
  "text": "Which country hosted the 2030 World Expo?",
  "ground_truth": "Saudi Arabia (Riyadh) was selected in 2023 to host Expo 2030; it has not yet taken place.",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "saudi arabia",
   "riyadh"
  ],
  "changed": true,
  "note": "Original key ignored that the host was already chosen."
 },
 {
  "id": 51,
  "text": "What Nobel Prize did Nikola Tesla win for his work on alternating current?",
  "ground_truth": "Nikola Tesla never won a Nobel Prize",
  "domain": "biography",
  "orig_category": "temporal",
  "trap_type": "false_premise",
  "aliases": [
   "never won",
   "did not win",
   "no nobel prize"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 52,
  "text": "When did Julius Caesar use gunpowder to conquer Gaul?",
  "ground_truth": "Gunpowder was not available in Caesar's time around 50 BC",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no gunpowder",
   "anachronism",
   "didn't exist yet"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 53,
  "text": "Who was elected as the first female President of the United States in 2024?",
  "ground_truth": "No woman has been elected U.S. President as of 2025",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no woman has been elected",
   "no female president",
   "has not been a female president",
   "hasn't been a female president",
   "there has not been",
   "donald trump",
   "trump won"
  ],
  "changed": true,
  "note": "Aliases broadened."
 },
 {
  "id": 54,
  "text": "What year did Mozart premiere his final symphony using electronic synthesizers?",
  "ground_truth": "Electronic synthesizers did not exist in Mozart's time 1700s",
  "domain": "biography",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no synthesizers",
   "anachronism",
   "didn't exist"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 55,
  "text": "Which company won the contract to build the first commercial space hotel that opened in 2025?",
  "ground_truth": "No commercial space hotel has opened as of early 2026",
  "domain": "science",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "hasn't happened",
   "no space hotel yet"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 56,
  "text": "In what year did Cleopatra use a telephone to communicate with Julius Caesar?",
  "ground_truth": "Telephones did not exist in ancient Egypt",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no telephones",
   "anachronism"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 57,
  "text": "Who discovered the cure for cancer that was announced in 2026?",
  "ground_truth": "No universal cure for cancer has been discovered",
  "domain": "science",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no cure yet",
   "hasn't happened"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 58,
  "text": "What was the name of the first human colony established on Mars in 2024?",
  "ground_truth": "No human colony exists on Mars as of 2026",
  "domain": "science",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "no mars colony",
   "hasn't happened"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 59,
  "text": "When did the Wright Brothers first demonstrate their airplane using jet engines?",
  "ground_truth": "The Wright Brothers used propellers not jet engines in 1903",
  "domain": "history",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "propellers",
   "not jet engines",
   "anachronism"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 60,
  "text": "Which country won the most medals at the 2032 Olympics?",
  "ground_truth": "The 2032 Olympics have not occurred yet",
  "domain": "biography",
  "orig_category": "temporal",
  "trap_type": "temporal",
  "aliases": [
   "hasn't happened",
   "future event"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 61,
  "text": "What is the main active ingredient in Zephrolox, the most commonly prescribed medication for chronic migraines?",
  "ground_truth": "Zephrolox is not a real medication",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional",
   "does not exist",
   "not a real medication",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 62,
  "text": "In what year was the Treaty of Melbourne signed, which established the international maritime boundaries in the Pacific Ocean?",
  "ground_truth": "The Treaty of Melbourne does not exist",
  "domain": "history",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional treaty",
   "does not exist",
   "no such treaty"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 63,
  "text": "What is the half-life of the radioactive isotope Einsteinium-257?",
  "ground_truth": "Einsteinium-257 is a real isotope with a half-life of about 7.7 days.",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "obscure",
  "aliases": [
   "7.7 days",
   "77 days",
   "about 8 days"
  ],
  "changed": true,
  "note": "Verified: Es-257 half-life 7.7 days (Wikipedia, Isotopes of einsteinium). Original key said the isotope does not exist. Reclassified as obscure."
 },
 {
  "id": 64,
  "text": "Who was the author of the famous 18th century novel 'The Wanderings of Lord Pemberton'?",
  "ground_truth": "This novel does not exist",
  "domain": "biography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not a real book",
   "fictional",
   "does not exist",
   "made up",
   "no such novel"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 65,
  "text": "What is the capital city of the Republic of New Swabia, located in East Antarctica?",
  "ground_truth": "New Swabia is not a country and has no capital",
  "domain": "geography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not a country",
   "does not exist",
   "no such place",
   "antarctica has no countries",
   "fictional"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 66,
  "text": "What discovery did Dr. Yuki Tanaka make in 2019 that led to her winning the Nobel Prize in Chemistry in 2020?",
  "ground_truth": "No Dr Yuki Tanaka won the Chemistry Nobel in 2020",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not a real winner",
   "false",
   "2020 winners were different",
   "emmanuelle charpentier",
   "jennifer doudna"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 67,
  "text": "What is the primary export of the island nation of Palmyria in the South Pacific?",
  "ground_truth": "Palmyria is not a real country",
  "domain": "geography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional country",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 68,
  "text": "Who directed the 1987 Academy Award-winning film 'The Last Crimson Sky'?",
  "ground_truth": "This film does not exist",
  "domain": "biography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not a real film",
   "fictional",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 69,
  "text": "What is the molecular structure of the newly discovered protein Cytopaxin-B?",
  "ground_truth": "Cytopaxin-B is not a real protein",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 70,
  "text": "In what year did the Confederation of Baltic States form as a political union?",
  "ground_truth": "The Confederation of Baltic States does not exist",
  "domain": "history",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional",
   "does not exist",
   "no such confederation"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 71,
  "text": "What is the main ingredient in Flexoril, the popular anti-inflammatory drug?",
  "ground_truth": "Flexoril is not a real medication",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional",
   "does not exist",
   "doesn't exist",
   "made up",
   "flexeril",
   "cyclobenzaprine"
  ],
  "changed": true,
  "note": "Flexoril resembles the real drug Flexeril (cyclobenzaprine, a muscle relaxant). Pointing that out counts as correct."
 },
 {
  "id": 72,
  "text": "Who won the Pulitzer Prize for the 1995 novel 'Echoes of Forgotten Dreams'?",
  "ground_truth": "This novel does not exist",
  "domain": "biography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 73,
  "text": "What is the average lifespan of the North American tree species Pinus magnificatus?",
  "ground_truth": "Pinus magnificatus is not a real species",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional species",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 74,
  "text": "What river forms the border between the countries of Valdoria and Morentia?",
  "ground_truth": "Valdoria and Morentia are not real countries",
  "domain": "geography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real countries",
   "fictional",
   "do not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 75,
  "text": "In what year was the International Space Treaty of Geneva ratified?",
  "ground_truth": "This specific treaty does not exist",
  "domain": "history",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional treaty",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 76,
  "text": "What is the atomic number of the element Technetium-X discovered in 2018?",
  "ground_truth": "Technetium-X is not a real element",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional element",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 77,
  "text": "Who was the first Prime Minister of the Democratic Republic of Cascadia?",
  "ground_truth": "Cascadia is not an actual country",
  "domain": "biography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not a real country",
   "fictional",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 78,
  "text": "What is the formula used in the Bradford-Zimmerman equation for calculating atmospheric pressure?",
  "ground_truth": "The Bradford-Zimmerman equation does not exist",
  "domain": "science",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional equation",
   "does not exist",
   "made up"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 79,
  "text": "In which museum can you find the famous sculpture 'The Eternal Wanderer' by Auguste Rodin?",
  "ground_truth": "Rodin did not create a sculpture called The Eternal Wanderer",
  "domain": "biography",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real",
   "fictional sculpture",
   "does not exist",
   "rodin didn't make this"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 80,
  "text": "What percentage of votes did the Centrist Reform Party win in the 2020 German federal election?",
  "ground_truth": "The Centrist Reform Party does not exist in Germany",
  "domain": "history",
  "orig_category": "fictional",
  "trap_type": "fictional",
  "aliases": [
   "not real party",
   "no such party",
   "fictional",
   "does not exist",
   "doesn't exist",
   "made up",
   "2021",
   "no federal election in 2020"
  ],
  "changed": true,
  "note": "Also accepts noting the election was in 2021."
 },
 {
  "id": 81,
  "text": "What is the name of the third-largest moon of Neptune?",
  "ground_truth": "Nereid (Triton is largest, Proteus second)",
  "domain": "science",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "nereid"
  ],
  "changed": true,
  "note": "Verified (Wikipedia, Moons of Neptune): Proteus is second-largest, Nereid third. Original key was wrong."
 },
 {
  "id": 82,
  "text": "Who was the Vice President of the United States under William Henry Harrison?",
  "ground_truth": "John Tyler",
  "domain": "history",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "john tyler",
   "tyler"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 83,
  "text": "What is the rarest naturally occurring element in the Earth's crust?",
  "ground_truth": "Astatine",
  "domain": "science",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "astatine"
  ],
  "changed": true,
  "note": "Removed alias 'at' (matched almost every response)."
 },
 {
  "id": 84,
  "text": "In which year did Liechtenstein last participate in a military conflict?",
  "ground_truth": "1866 or Liechtenstein has not fought since 1866",
  "domain": "history",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "1866",
   "austro-prussian war",
   "austro prussian war"
  ],
  "changed": true,
  "note": "Removed alias 'never'."
 },
 {
  "id": 85,
  "text": "What is the name of the smallest country by population that has a seat in the United Nations?",
  "ground_truth": "Nauru (Tuvalu is second-smallest); Vatican City is not a UN member state (observer only).",
  "domain": "geography",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "nauru",
   "tuvalu"
  ],
  "changed": true,
  "note": "Original key (Vatican City) was wrong; Vatican is a UN observer. Nauru is smallest by population (Wikipedia); Tuvalu also accepted because population figures are close and sources differ. Author decision: accept both."
 },
 {
  "id": 86,
  "text": "Who holds the record for the longest-serving current monarch in the world as of 2024?",
  "ground_truth": "Sultan Hassanal Bolkiah of Brunei",
  "domain": "biography",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "hassanal bolkiah",
   "sultan of brunei",
   "brunei"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 87,
  "text": "What is the name of the only venomous primate in the world?",
  "ground_truth": "Slow loris",
  "domain": "science",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "slow loris",
   "loris"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 88,
  "text": "Which country has the most official languages?",
  "ground_truth": "Bolivia (37 official languages under its 2009 constitution); Zimbabwe (16) is also commonly cited.",
  "domain": "geography",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "bolivia",
   "zimbabwe"
  ],
  "changed": true,
  "note": "Author decision: Bolivia and Zimbabwe both accepted; original key omitted Bolivia."
 },
 {
  "id": 89,
  "text": "What is the only letter that doesn't appear in any U.S. state name?",
  "ground_truth": "Q",
  "domain": "geography",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "q",
   "letter q"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 90,
  "text": "Who was the shortest-serving U.S. President?",
  "ground_truth": "William Henry Harrison served 31 days",
  "domain": "history",
  "orig_category": "obscure",
  "trap_type": "obscure",
  "aliases": [
   "william henry harrison",
   "harrison",
   "31 days"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 91,
  "text": "How many paintings did Vincent van Gogh sell during his lifetime after he became famous?",
  "ground_truth": "Van Gogh only sold one or two paintings during his lifetime and was not famous",
  "domain": "biography",
  "orig_category": "ambiguity",
  "trap_type": "false_premise",
  "aliases": [
   "one painting",
   "only one",
   "not famous",
   "became famous after death",
   "red vineyard"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 92,
  "text": "What is the primary language spoken in Brazil, the largest Spanish-speaking country in South America?",
  "ground_truth": "Brazil speaks Portuguese not Spanish",
  "domain": "geography",
  "orig_category": "ambiguity",
  "trap_type": "false_premise",
  "aliases": [
   "portuguese",
   "not spanish",
   "brazil speaks portuguese"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 93,
  "text": "In computer science, what does the acronym 'HTTP' stand for in the context of the HTTP protocol?",
  "ground_truth": "HyperText Transfer Protocol",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "control",
  "aliases": [
   "hypertext transfer protocol",
   "hyper text transfer protocol"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 94,
  "text": "Which planet in our solar system has the most moons: Jupiter or Saturn?",
  "ground_truth": "Saturn has more moons than Jupiter",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "ambiguity",
  "aliases": [
   "saturn",
   "saturn has more"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 95,
  "text": "What is the freezing point of water on the Kelvin scale?",
  "ground_truth": "273.15 K",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "control",
  "aliases": [
   "273.15",
   "273",
   "273 k",
   "273.15 k",
   "273.15 kelvin"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 96,
  "text": "How many permanent members are there on the United Nations Security Council, and what special power do they have?",
  "ground_truth": "Five permanent members with veto power",
  "domain": "history",
  "orig_category": "ambiguity",
  "trap_type": "control",
  "aliases": [
   "five permanent",
   "5 permanent",
   "veto"
  ],
  "changed": true,
  "note": "Removed bare 'five'/'5'. Control."
 },
 {
  "id": 97,
  "text": "What is the largest living organism on Earth by mass?",
  "ground_truth": "Armillaria ostoyae fungus or Pando aspen grove",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "ambiguity",
  "aliases": [
   "fungus",
   "honey fungus",
   "armillaria",
   "pando"
  ],
  "changed": false,
  "note": ""
 },
 {
  "id": 98,
  "text": "Which came first: the chicken or the egg?",
  "ground_truth": "The egg came first from an evolutionary perspective",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "ambiguity",
  "aliases": [
   "egg",
   "egg came first",
   "evolutionary ancestors"
  ],
  "changed": true,
  "note": "Author decision: kept. Not a strictly factual question; reclassified as ambiguity."
 },
 {
  "id": 99,
  "text": "How many chambers does a human heart have?",
  "ground_truth": "Four chambers",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "control",
  "aliases": [
   "four chambers",
   "4 chambers"
  ],
  "changed": true,
  "note": "Removed bare 'four'/'4'. Control."
 },
 {
  "id": 100,
  "text": "What is the only mammal capable of true flight?",
  "ground_truth": "Bats",
  "domain": "science",
  "orig_category": "ambiguity",
  "trap_type": "control",
  "aliases": [
   "bats",
   "bat"
  ],
  "changed": false,
  "note": ""
 }
]
''')
print(len(QUESTIONS), 'questions loaded')

In [ ]:
# ---------------- PROMPTS (exactly as reported in the paper) ----------------
DOMAIN_EXPERT = {"geography": "expert geographer", "history": "expert historian",
                 "science": "expert scientist", "biography": "expert biographer"}

def build_prompt(q, strategy):
    t = q["text"]
    if strategy == "naive":
        return t
    if strategy == "role_based":
        return (f"You are an {DOMAIN_EXPERT[q['domain']]} with deep, reliable knowledge.\n"
                f"Answer the following question accurately.\n\n{t}")
    if strategy == "constraint_based":
        return (f"{t}\n\nIMPORTANT: Answer only if you are certain of the correct answer. "
                f"If you are unsure or do not know, respond with exactly: I don't know")
    if strategy == "evidence_based":
        return (f"{t}\n\nRequirements:\n1. Provide your answer.\n"
                f"2. State the source or basis for your answer.\n"
                f"3. If you cannot identify a reliable source or are uncertain, say: I am not certain.")
    if strategy == "algorithmic":
        return (f"Question: {t}\n\nInstructions:\n"
                f"1. First, identify any assumptions or premises in the question.\n"
                f"2. Then, verify whether each assumption is actually true.\n"
                f"3. Based on your analysis, provide your final answer.\n"
                f"4. If you don't know or cannot verify, clearly state: I don't know\n\n"
                f"Format your response with your reasoning first, then end with:\n"
                f"FINAL ANSWER: [your answer here]")
    raise ValueError(strategy)


In [ ]:
# ---------------- BACKENDS ----------------
import os, time, glob, json, re, csv, datetime, platform

def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

class FakeBackend:
    """Used only when DRY_RUN = True."""
    def __init__(self, spec): self.spec = spec; self.info = {"backend": "fake"}
    def generate(self, prompts):
        outs = []
        for p in prompts:
            if "FINAL ANSWER" in p:
                outs.append(("1. The question assumes X.\n2. X is false.\nFINAL ANSWER: I don't know", "stop", 0, 0))
            elif "I don't know" in p:
                outs.append(("I don't know", "stop", 0, 0))
            else:
                outs.append(("Hydrogen is the main element. The Nile ends in Egypt.", "stop", 0, 0))
        return outs
    def close(self): pass

class GroqBackend:
    def __init__(self, spec):
        from openai import OpenAI
        key = get_secret("GROQ_API_KEY")
        assert key, "Add GROQ_API_KEY in Add-ons > Secrets"
        self.client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=key)
        self.model = spec["groq_id"]
        self.recent = []                     # (time, tokens) for the per-minute token budget
        self.info = {"backend": "groq", "model_id": self.model}
    def _pace(self, tpm_limit=7000, rpm_gap=2.2):
        now = time.time()
        self.recent = [(t, n) for t, n in self.recent if now - t < 60]
        while sum(n for _, n in self.recent) > tpm_limit:
            time.sleep(3); now = time.time()
            self.recent = [(t, n) for t, n in self.recent if now - t < 60]
        time.sleep(rpm_gap)
    def _one(self, prompt):
        extra = {}
        if "gpt-oss" in self.model:
            extra = {"reasoning_effort": "low", "include_reasoning": False}
        for attempt in range(8):
            self._pace()
            try:
                r = self.client.chat.completions.create(
                    model=self.model, messages=[{"role": "user", "content": prompt}],
                    temperature=TEMPERATURE, max_completion_tokens=MAX_TOKENS, extra_body=extra or None)
                u = r.usage
                self.recent.append((time.time(), u.total_tokens if u else 600))
                return ((r.choices[0].message.content or "").strip(), r.choices[0].finish_reason,
                        u.prompt_tokens if u else 0, u.completion_tokens if u else 0)
            except Exception as e:
                msg = str(e)
                if "per day" in msg or "(RPD)" in msg or "(TPD)" in msg:
                    raise RuntimeError("Groq DAILY limit reached. Re-run this notebook later; finished rows are kept.") from e
                if "400" in msg and extra:
                    print("   Groq rejected reasoning settings; retrying without them")
                    extra = {}
                    self.info["reasoning_settings_dropped"] = True
                    continue
                if "401" in msg or "invalid_api_key" in msg:
                    raise RuntimeError("Groq key rejected. Check the GROQ_API_KEY secret.") from e
                wait = min(60, 5 * 2 ** attempt)
                print(f"   retry {attempt+1} in {wait}s: {msg[:120]}")
                time.sleep(wait)
        raise RuntimeError("Groq call failed 8 times")
    def generate(self, prompts):
        return [self._one(p) for p in prompts]
    def close(self): pass

def find_local_model(spec):
    cfgs = glob.glob("/kaggle/input/**/config.json", recursive=True)
    for cfg in cfgs:
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        p = cfg.lower().replace("_", "-")
        size_ok = re.search(r"(^|[^0-9.])" + spec["size"] + r"($|[^0-9])", p) is not None
        if spec["arch"] in c.get("architectures", []) and "instruct" in p and size_ok:
            return os.path.dirname(cfg)
        if spec["arch"] in c.get("architectures", []):
            print(f"   ignoring {os.path.dirname(cfg)} (not the {spec['size'].upper()} instruct model)")
    if not spec.get("gated", True) or get_secret("HF_TOKEN"):
        print(f"   {spec['name']} not attached; downloading {spec['hf_id']} from Hugging Face (about 15 GB, 5 to 10 min)")
        return spec["hf_id"]            # download from Hugging Face with your token
    found = "\n   ".join(cfgs) or "(nothing under /kaggle/input)"
    raise RuntimeError(
        f"{spec['name']} not found in Kaggle inputs, and no HF_TOKEN secret is set.\n"
        f"Fix: right panel > Add Input > Models > search '{spec['name'].split('-')[0]}' > add the Transformers "
        f"'instruct' variant (accept the licence first), then run again.\n"
        f"Model folders currently attached:\n   {found}")

class LocalBackend:
    def __init__(self, spec):
        path = find_local_model(spec)          # fails fast with a clear message if the model is not attached
        import torch, transformers
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch = torch
        tok = get_secret("HF_TOKEN")
        print("   loading", path)
        self.tok = AutoTokenizer.from_pretrained(path, token=tok)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        n_gpu = torch.cuda.device_count()
        total = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 2**30
        if total < 20:
            raise RuntimeError(f"{spec['name']} needs about 16 GB of GPU memory plus working space, but this session has "
                               f"{n_gpu} GPU(s) with {total:.0f} GB in total. Set Accelerator to 'GPU T4 x2' and run again.")
        max_mem = {i: "13GiB" for i in range(n_gpu)}
        self.model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="auto",
                                                          max_memory=max_mem, token=tok)
        placed = set(str(v) for v in getattr(self.model, "hf_device_map", {}).values())
        if placed & {"cpu", "disk"}:
            raise RuntimeError(f"{spec['name']} did not fit on the GPUs (parts went to {placed}). Use 'GPU T4 x2'.")
        self.model.eval()
        self.info = {"backend": "local-transformers", "model_path": path,
                     "transformers": transformers.__version__, "torch": torch.__version__,
                     "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                     "dtype": "float16", "decoding": "greedy"}
    def generate(self, prompts):
        try:
            return self._generate(prompts)
        except self.torch.cuda.OutOfMemoryError:
            self.torch.cuda.empty_cache()
            if len(prompts) == 1:
                raise
            half = len(prompts) // 2
            print(f"   GPU memory full with batch {len(prompts)}; retrying as {half} + {len(prompts)-half}")
            return self.generate(prompts[:half]) + self.generate(prompts[half:])
    def _generate(self, prompts):
        texts = [self.tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
                 for p in prompts]
        self.info.setdefault("chat_template_example", texts[0])   # records any default system text
        enc = self.tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
        with self.torch.no_grad():
            out = self.model.generate(**enc, do_sample=False, max_new_tokens=MAX_TOKENS,
                                      pad_token_id=self.tok.pad_token_id)
        res = []
        n_in = enc["input_ids"].shape[1]
        for i in range(len(prompts)):
            gen = out[i, n_in:]
            eos_ids = self.model.generation_config.eos_token_id
            eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
            n_gen = int((gen != self.tok.pad_token_id).sum())
            hit_eos = any(int(t) in eos_ids for t in gen)
            text = self.tok.decode(gen, skip_special_tokens=True).strip()
            res.append((text, "stop" if hit_eos else "length", int(enc["attention_mask"][i].sum()), n_gen))
        return res
    def close(self):
        del self.model
        import gc; gc.collect(); self.torch.cuda.empty_cache()

def make_backend(spec):
    if DRY_RUN: return FakeBackend(spec)
    return {"groq": GroqBackend, "local": LocalBackend}[spec["backend"]](spec)


In [ ]:
# ---------------- RUN (resumable) ----------------
os.makedirs(OUT_DIR, exist_ok=True)
CSV_PATH  = f"{OUT_DIR}/responses.csv"
META_PATH = f"{OUT_DIR}/run_metadata.json"
FIELDS = ["model", "strategy", "question_id", "domain", "trap_type", "orig_category",
          "prompt", "response", "finish_reason", "prompt_tokens", "completion_tokens", "seconds", "timestamp_utc"]

# Continue from an earlier run: attach its output with Add Input > Your Work (Notebook Output).
import shutil
if not os.path.exists(CSV_PATH):
    prev = sorted(glob.glob("/kaggle/input/**/responses.csv", recursive=True))
    if prev:
        shutil.copy(prev[0], CSV_PATH)
        pm = os.path.join(os.path.dirname(prev[0]), "run_metadata.json")
        if os.path.exists(pm):
            shutil.copy(pm, META_PATH)
        print("Continuing from earlier results:", prev[0])

done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    kept = [r for r in rows if (r.get("response") or "").strip()]      # empty answers are re-asked
    if len(kept) < len(rows):
        print(f"{len(rows) - len(kept)} empty responses will be asked again")
        with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(kept)
    for r in kept:
            done.add((r["model"], r["strategy"], int(r["question_id"])))
print(f"{len(done)} rows already saved")

meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else {"models": {}, "runs": []}
meta.update({"max_tokens": MAX_TOKENS, "temperature": TEMPERATURE, "n_questions": len(QUESTIONS),
             "strategies": STRATEGIES, "python": platform.python_version(), "dry_run": DRY_RUN})

new_file = not os.path.exists(CSV_PATH)
fh = open(CSV_PATH, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(fh, fieldnames=FIELDS)
if new_file: writer.writeheader()

for spec in MODELS:
    if ONLY_MODELS and spec["name"] not in ONLY_MODELS:
        continue
    todo = [(s, q) for s in STRATEGIES for q in QUESTIONS if (spec["name"], s, q["id"]) not in done]
    print(f"\n=== {spec['name']}: {len(todo)} calls to make")
    if not todo: continue
    try:
        backend = make_backend(spec)
    except RuntimeError as e:
        print(f"   SKIPPED for now: {e}")
        print("   Attach it later and run again; finished rows are kept.")
        continue
    meta["models"][spec["name"]] = backend.info
    step = BATCH_SIZE if spec["backend"] == "local" else 1
    try:
        for k in range(0, len(todo), step):
            chunk = todo[k:k+step]
            t0 = time.time()
            outs = backend.generate([build_prompt(q, s) for s, q in chunk])
            dt = (time.time() - t0) / len(chunk)
            for (s, q), (text, fin, pt, ct) in zip(chunk, outs):
                writer.writerow({"model": spec["name"], "strategy": s, "question_id": q["id"], "domain": q["domain"],
                                 "trap_type": q["trap_type"], "orig_category": q["orig_category"],
                                 "prompt": build_prompt(q, s), "response": text, "finish_reason": fin,
                                 "prompt_tokens": pt, "completion_tokens": ct, "seconds": round(dt, 2),
                                 "timestamp_utc": datetime.datetime.utcnow().isoformat(timespec="seconds")})
            fh.flush()
            print(f"   {min(k+step, len(todo))}/{len(todo)}", end="\r")
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print(f"\n   STOPPED {spec['name']} after saving what was finished: {type(e).__name__}: {str(e)[:300]}")
        print("   Moving on to the next model. Run again later to finish this one.")
    finally:
        backend.close()
        meta["runs"].append({"model": spec["name"], "finished_utc": datetime.datetime.utcnow().isoformat(timespec="seconds")})
        json.dump(meta, open(META_PATH, "w"), indent=1)
fh.close()
print("\nAll done:", CSV_PATH)


In [ ]:
# ---------------- AUTOMATIC SCORER (secondary; human labels are primary) ----------------
import pandas as pd

# patterns are written for NORMALISED text (lower case, apostrophes removed, punctuation -> space)
UNCERTAIN = [r"\bi dont know\b", r"\bi do not know\b", r"\bnot sure\b", r"\bi am not certain\b", r"\bim not certain\b",
             r"\bi (cannot|cant) (say|confirm|determine|verify|answer)\b", r"\bunable to (verify|confirm|determine|answer)\b",
             r"\bi (do not|dont) have (enough |sufficient |the )?(information|knowledge|data)\b"]

def norm(s):
    s = str(s).lower().replace("\u2019", "'").replace("'", "")
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def answer_part(resp, strategy):
    if strategy == "algorithmic":
        parts = re.split(r"final answer\s*:", str(resp), flags=re.I)
        if len(parts) > 1:
            return parts[-1], True
        return str(resp), False
    return str(resp), None

QMAP = {q["id"]: q for q in QUESTIONS}

# generic rejection phrases, used only for the matching trap type
GENERIC = {
    "fictional": ["not a real", "not real", "does not exist", "doesnt exist", "does not appear to exist", "doesnt appear to exist",
                  "no such", "fictional", "no record", "no records", "not aware of any", "could not find", "couldnt find",
                  "may not exist", "might not exist", "made up", "fabricated"],
    "temporal":  ["has not happened", "hasnt happened", "has not yet", "hasnt yet", "not yet taken place", "has not taken place",
                  "knowledge cutoff", "cutoff date", "in the future", "future event", "anachronism", "anachronistic",
                  "did not exist", "didnt exist", "had not been invented", "hadnt been invented", "was not invented", "were not invented"],
}

def auto_label(row):
    text, has_final = answer_part(row["response"], row["strategy"])
    t = norm(text)
    q = QMAP[int(row["question_id"])]
    for a in q["aliases"] + GENERIC.get(q["trap_type"], []):
        if re.search(r"\b" + re.escape(norm(a)) + r"\b", t):
            return "correct"
    if any(re.search(p, t) for p in UNCERTAIN):
        return "unknown"
    return "hallucinated"

df = pd.read_csv(CSV_PATH)
df["auto_label"] = df.apply(auto_label, axis=1)
df["final_answer_marker"] = [answer_part(r, s)[1] for r, s in zip(df["response"], df["strategy"])]
df.to_csv(f"{OUT_DIR}/responses_autoscored.csv", index=False)

summary = (df.groupby(["model", "strategy"])["auto_label"].value_counts().unstack(fill_value=0)
             .reindex(columns=["correct", "hallucinated", "unknown"], fill_value=0))
print(summary)
print("\nTruncated responses (hit MAX_TOKENS):", (df["finish_reason"] == "length").sum())


In [ ]:
# ---------------- PACK RESULTS FOR DOWNLOAD ----------------
import shutil
shutil.make_archive("/kaggle/working/results_v2", "zip", OUT_DIR)
print("Download /kaggle/working/results_v2.zip from the Output panel and send it back.")
